# Medical Image Generation with GANs
This is the modified version of the previous notebook that contains the tpu setup with generator and discriminator models.

In [1]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.5 MB/s eta 0:00:00


## Import Libraries

In [2]:
import tensorflow as tf
import numpy as np
import os
import math
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.data import TFRecordDataset
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Reshape, Conv2D, Conv2DTranspose
from tensorflow.keras.optimizers import Adam
from google.colab import drive

Since we will be using TFRecords for storing and processing of the data, it is advised to use TPUs; however, they are not always available

In [3]:
import tensorflow as tf

try:
    # Detect TPU
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print("Running on TPU:", tpu.master())

    # Connect and initialize TPU system
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)

    # Create TPU strategy
    strategy = tf.distribute.TPUStrategy(tpu)
except ValueError:
    # If TPU is not found, fallback to CPU/GPU
    strategy = tf.distribute.get_strategy()
    print("Running on CPU/GPU")

# Display the strategy used
print("Using strategy:", strategy)

Running on CPU/GPU
Using strategy: <tensorflow.python.distribute.distribute_lib._DefaultDistributionStrategy object at 0x79b3ceb4c290>


In [7]:
print(tf.config.list_logical_devices('TPU'))

[]


Else, check if tensorflow recognizes GPU. If its zero it means you're running on CPU

## Data Import and Setup
Data is being fetched from kaggle and unzipped using linux `unzip` command. The downloaded dataset is converted to `TFRecordDataset` and saved as `brst_dataset.tfrecord` file.

Note: Kaggle credentials are required to access the dataset.

Here are the steps for accessing kaggle:
1. Create `kaggle.json` file in colab
2. Add your credentials to json file as follows:
```json
{
    "username": "YOUR_USER_NAME",
    "key": "YOUR_API_KEY"
}
```

After following these steps, run the next cells to download and setup the dataset.

In [8]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"rumiyyaalili","key":"5a48756827bd010b7b5a1f5561da34e8"}'}

In [10]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [11]:
!kaggle datasets download -d rumiyyaalili/brst-dataset-binary

Dataset URL: https://www.kaggle.com/datasets/rumiyyaalili/brst-dataset-binary
License(s): unknown


In [13]:
!unzip -qq brst-dataset-binary

In [14]:
os.makedirs("images", exist_ok=True)

In the below cell we can see the raw content of a tfrecord

In [15]:
raw_dataset = tf.data.TFRecordDataset(["brst_dataset.tfrecord"])
for raw_record in raw_dataset.take(1):  # Take a few records to inspect
    example = tf.train.Example()
    example.ParseFromString(raw_record.numpy())
    print(example)

features {
  feature {
    key: "image_raw"
    value {
      bytes_list {
        value: "\211PNG\r\n\032\n\000\000\000\rIHDR\000\000\001\000\000\000\001\000\010\000\000\000\000y\031\367\272\000\000;\370IDATx\234\345\275Y\314\256Yv\337\265\246\275\237\341\035\277\341\314\347\324\251\271k\352n\267\333m;\335I:\301\031L\220b\004Q\202\204\242\220+$\244HD\344\n\t\205\\000\022BB(\227\334\020\205I\026\204\020)$1N\020\002\343\304\tq\333\335n\273z\250\252\256Su\306o|\247\347\331\303Z\213\213\352\030\223\020\023w\350\256\367\353\372_\275\257>\351\321\373\374\276\265\367Z{\355\275\326\006\370\204\013?\356\037\360;\320-\031\232JN\n\220\340\232L\254YGL\245c\360\260\201~GR`RiR^\264\247\341N\233`i\267\327w\276q\031\247\263\322>j\246\364\374%\034l|\372\370\325\017\257=\232\276\366\365\033\327\272\311\342*\001\270\326\024R\202P\364\360Y\355'M\320.\325\032\254\370,\214\343\2444\331Y\211\244\321Y?\320+72\237M\007\257\335z\261\355\350\350$\336~\372\334E\203\007\273\246\366\003\236\337\273\271\273\325Rw\2

For persistency, we will save the weights and generated images in google drive.

In [17]:
# Mount Google Drive
drive.mount('/content/drive')


# Define directories inside Google Drive
base_image_dir = "/content/drive/My Drive/TF_GAN_Generated_Images"
save_dir = "/content/drive/My Drive/TF_GAN_Weights"
os.makedirs(base_image_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

Mounted at /content/drive


## Helper functions
We introduce serveral functions for simplifying the main logic of training.
These functions will be used for parsing, loading, storing, generating, etc.

In [18]:
# Load TFRecord Dataset
def load_tfrecord_dataset(tfrecord_file, batch_size):
    dataset = tf.data.TFRecordDataset([tfrecord_file])
    dataset = dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

In [19]:
def _parse_function(proto):
    keys_to_features = {'image_raw': tf.io.FixedLenFeature([], tf.string)}
    parsed_features = tf.io.parse_single_example(proto, keys_to_features)

    # Ensure channels=1 for grayscale
    image = tf.io.decode_png(parsed_features['image_raw'], channels=1)
    # Normalize to [-1, 1]
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    # Resize to the expected shape
    image = tf.image.resize(image, img_shape[:2])

    return image

In [20]:
def generate_and_display_images(generator, num_images=25, latent_dim=100, nrow=5):
    # Generate latent vectors
    z = tf.random.normal([num_images, latent_dim])  # Shape: (num_images, latent_dim)

    # Generate images
    generated_images = generator(z, training=False)  # Shape: (num_images, height, width, channels)

    # Convert tensor to NumPy for visualization
    generated_images = generated_images.numpy()

    # Normalize to [0, 1] for display
    generated_images = (generated_images + 1) / 2.0

    # Determine grid size
    fig, axes = plt.subplots(nrow, nrow, figsize=(10, 10))
    for i, ax in enumerate(axes.flat):
        if i < num_images:
            ax.imshow(generated_images[i, :, :, 0], cmap="gray")  # Grayscale image
            ax.axis("off")
    plt.show()

In [21]:
# Save Generated Images
def save_generated_images(generator, epoch, num_images=25):
    z = tf.random.normal([num_images, latent_dim])
    generated_images = generator(z, training=False)
    for i in range(num_images):
        img_path = os.path.join(base_image_dir, f"epoch_{epoch}_img_{i}.png")
        tf.keras.preprocessing.image.save_img(img_path, (generated_images[i].numpy() + 1) * 127.5)  # Rescale
    print(f"Saved generated images for epoch {epoch}.")

## Training
The model type is GAN with Gradient Penalty. It has Generator and the Discriminator components. It uses ADAM optimizer and Discriminator/Generator Loss functions.



### Models

In [22]:
class Generator(tf.keras.Model):
    def __init__(self, latent_dim, img_shape):
        super(Generator, self).__init__()
        self.img_shape = img_shape  # (height, width, channels)
        self.height = img_shape[0]
        self.width = img_shape[1]
        self.channels = int(img_shape[2])  # Explicitly convert to int

        def block(units, normalize=True):
            block_layers = [layers.Dense(units)]
            if normalize:
                block_layers.append(layers.BatchNormalization(momentum=0.8))
            block_layers.append(layers.LeakyReLU(negative_slope=0.2))  # Updated parameter name
            return block_layers

        # Fully connected layers
        self.model_fc = tf.keras.Sequential(
            block(128, normalize=False) +
            block(256) +
            block(512) +
            block(1024) +
            [layers.Dense(int(tf.math.reduce_prod(img_shape)), activation="tanh")]  # Explicitly cast to int
        )

        # Convolutional refinement part
        self.conv_refine = tf.keras.Sequential([
            layers.Conv2D(32, kernel_size=3, strides=1, padding="same"),
            layers.LeakyReLU(negative_slope=0.2),  # Updated parameter name
            layers.Conv2D(32, kernel_size=3, strides=1, padding="same"),
            layers.LeakyReLU(negative_slope=0.2),  # Updated parameter name
            layers.Conv2D(self.channels, kernel_size=3, strides=1, padding="same", activation="tanh"),
        ])

    def call(self, z, training=False):  # Added training parameter for consistency
        # Generate image from latent vector
        img = self.model_fc(z)

        # Reshape into (batch_size, height, width, channels)
        img = tf.reshape(img, (-1, self.height, self.width, self.channels))

        # Apply CNN refinement
        img = self.conv_refine(img, training=training)

        return img

In [23]:
class Discriminator(tf.keras.Model):
    def __init__(self, img_shape):
        super(Discriminator, self).__init__()
        self.img_shape = img_shape  # (height, width, channels)

        self.conv_layers = tf.keras.Sequential([
            layers.Conv2D(64, kernel_size=4, strides=2, padding="same"),
            layers.LeakyReLU(negative_slope=0.2),  # Updated parameter name

            layers.Conv2D(128, kernel_size=4, strides=2, padding="same"),
            layers.LeakyReLU(negative_slope=0.2),  # Updated parameter name

            layers.Conv2D(256, kernel_size=4, strides=2, padding="same"),
            layers.LeakyReLU(negative_slope=0.2),  # Updated parameter name

            layers.Conv2D(512, kernel_size=4, strides=2, padding="same"),
            layers.LeakyReLU(negative_slope=0.2),  # Updated parameter name
        ])

        self.global_pool = layers.GlobalAveragePooling2D()  # Compute flattened size dynamically

        self.fc_layers = tf.keras.Sequential([
            layers.Dense(512),
            layers.LeakyReLU(negative_slope=0.2),  # Updated parameter name
            layers.Dense(1)  # Output a single scalar (real/fake)
        ])

    def call(self, img, training=False):  # Added training parameter for consistency
        features = self.conv_layers(img, training=training)  # Apply convolutional layers
        features_flat = self.global_pool(features)  # Flatten automatically
        validity = self.fc_layers(features_flat, training=training)  # Fully connected layers
        return validity

### Hyperparameters

In [24]:
checkpoint_interval = 10  # Save every 10 epochs
lr_g = 0.0002
lr_d = 0.0002
b1, b2 = 0.5, 0.999
latent_dim = 100
img_shape = (256, 256, 1)  # Make sure the shape matches the TFRecord images
lambda_gp = 10.0
batch_size = 64
epochs = 100
n_critic = 5  # Train discriminator 5x per generator step

### Training setup

In [25]:
# Instantiate the models
generator = Generator(latent_dim, img_shape)
discriminator = Discriminator(img_shape)

In [26]:
# Optimizers
optimizer_G = Adam(learning_rate=lr_g, beta_1=b1, beta_2=b2)
optimizer_D = Adam(learning_rate=lr_d, beta_1=b1, beta_2=b2)

In [27]:
# Gradient Penalty
def compute_gradient_penalty(D, real_images, fake_images):
    alpha = tf.random.uniform([real_images.shape[0], 1, 1, 1], 0.0, 1.0)
    interpolates = alpha * real_images + (1 - alpha) * fake_images
    with tf.GradientTape() as tape:
        tape.watch(interpolates)
        d_interpolates = D(interpolates, training=True)
    gradients = tape.gradient(d_interpolates, interpolates)
    gradients = tf.reshape(gradients, [gradients.shape[0], -1])
    gradient_penalty = tf.reduce_mean(tf.square(tf.norm(gradients, axis=1) - 1.0))
    return gradient_penalty

# Loss Functions
def discriminator_loss(real_validity, fake_validity, gradient_penalty):
    return -tf.reduce_mean(real_validity) + tf.reduce_mean(fake_validity) + lambda_gp * gradient_penalty

def generator_loss(fake_validity):
    return -tf.reduce_mean(fake_validity)

In [28]:
# Checkpointing
checkpoint = tf.train.Checkpoint(generator=generator, discriminator=discriminator,
                                 optimizer_G=optimizer_G, optimizer_D=optimizer_D)
checkpoint_manager = tf.train.CheckpointManager(checkpoint, directory=save_dir, max_to_keep=5)

# Restore checkpoint if available
if checkpoint_manager.latest_checkpoint:
    checkpoint.restore(checkpoint_manager.latest_checkpoint)
    print("Restored from latest checkpoint.")

In [29]:
# Training Function
def train(dataset, generator, discriminator, epochs, checkpoint_interval):
    for epoch in range(epochs):
        for i, batch in enumerate(dataset):
            real_images = batch

            # Train Discriminator
            with tf.GradientTape() as tape_D:
                z = tf.random.normal([real_images.shape[0], latent_dim])
                fake_images = generator(z, training=True)
                real_validity = discriminator(real_images, training=True)
                fake_validity = discriminator(fake_images, training=True)
                gradient_penalty = compute_gradient_penalty(discriminator, real_images, fake_images)
                d_loss = discriminator_loss(real_validity, fake_validity, gradient_penalty)
            gradients_D = tape_D.gradient(d_loss, discriminator.trainable_variables)
            optimizer_D.apply_gradients(zip(gradients_D, discriminator.trainable_variables))

            # Train Generator every n_critic steps
            if i % n_critic == 0:
                with tf.GradientTape() as tape_G:
                    z = tf.random.normal([real_images.shape[0], latent_dim])
                    fake_images = generator(z, training=True)
                    fake_validity = discriminator(fake_images, training=True)
                    g_loss = generator_loss(fake_validity)
                gradients_G = tape_G.gradient(g_loss, generator.trainable_variables)
                optimizer_G.apply_gradients(zip(gradients_G, generator.trainable_variables))

        print(f"[Epoch {epoch}/{epochs}] [D loss: {d_loss}] [G loss: {g_loss}]")

        # Save images & checkpoints
        if epoch % checkpoint_interval == 0:
            save_generated_images(generator, epoch)
            checkpoint_manager.save()
            print(f"Checkpoint saved at epoch {epoch}.")


### Train

In [30]:
# Load TFRecord Dataset
tfrecord_file = "/content/brst_dataset.tfrecord"
dataset = load_tfrecord_dataset(tfrecord_file, batch_size=batch_size)

# Start Training
with strategy.scope():
    train(dataset, generator, discriminator, epochs, checkpoint_interval)

Exception ignored in: <function _xla_gc_callback at 0x7995d3e4c2c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


KeyboardInterrupt: 